In [1]:
#@title install cloudscraper
!pip install cloudscraper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 3.6 MB/s eta 0:00:00


In [2]:
#@title ✔ script v1 dimakids
import os
import sys
import re
import json
import time
import math
import zipfile
import subprocess
import importlib.util
from pathlib import Path
from urllib.parse import urljoin

REQUIRED_PACKAGES = {
    "requests": "requests",
    "cloudscraper": "cloudscraper",
    "bs4": "beautifulsoup4",
    "rich": "rich",
    "requests_toolbelt": "requests-toolbelt",
}


def ensure_packages():
    missing = []
    for module_name, pip_name in REQUIRED_PACKAGES.items():
        if importlib.util.find_spec(module_name) is None:
            missing.append(pip_name)

    if missing:
        print(f"Installing missing packages: {', '.join(missing)}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *missing]
        )


ensure_packages()

import requests
import cloudscraper
from bs4 import BeautifulSoup
from rich.console import Console
from rich.prompt import Prompt
from requests_toolbelt.multipart.encoder import MultipartEncoder, MultipartEncoderMonitor


if os.name == "nt":
    try:
        os.system("")
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass


console = Console(highlight=False)
scraper = cloudscraper.create_scraper(browser={"browser": "chrome", "platform": "windows", "mobile": False})
CHUNK_SIZE = 256 * 1024
HIDDEN_FOLDERS = {"sample_data", ".ipynb_checkpoints", "__pycache__", ".git"}
BAR_WIDTH = 18


def is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


BASE_DIR = Path("/content") if is_colab() and Path("/content").exists() else Path.cwd()


def clean_name(name: str) -> str:
    name = re.sub(r'[\\/*?:"<>|]', "", (name or "").strip())
    return name or "Video"


def format_bytes(num):
    num = float(num)
    units = ["B", "KB", "MB", "GB", "TB"]
    for unit in units:
        if num < 1024 or unit == units[-1]:
            if unit == "B":
                return f"{int(num)}{unit}"
            return f"{num:.1f}{unit}"
        num /= 1024


def format_speed(num):
    return f"{format_bytes(num)}/s"


def format_eta(seconds):
    if seconds is None or seconds < 0 or math.isinf(seconds):
        return "--:--"
    seconds = int(seconds)
    mins, sec = divmod(seconds, 60)
    hrs, mins = divmod(mins, 60)
    if hrs > 0:
        return f"{hrs:02d}:{mins:02d}:{sec:02d}"
    return f"{mins:02d}:{sec:02d}"


def safe_print_line(text=""):
    sys.stdout.write(text + "\n")
    sys.stdout.flush()


def render_progress(prefix, current, total, start_time):
    elapsed = max(time.time() - start_time, 0.001)
    speed = current / elapsed

    if total and total > 0:
        percent = max(0.0, min(100.0, (current / total) * 100))
        filled = int(BAR_WIDTH * current / total)
        bar = "#" * filled + "-" * (BAR_WIDTH - filled)
        eta = (total - current) / speed if speed > 0 else None
        line = f"\r{prefix} [{bar}] {percent:5.1f}% {format_bytes(current)}/{format_bytes(total)} {format_speed(speed)} ETA {format_eta(eta)}"
    else:
        pulse = int((time.time() * 4) % (BAR_WIDTH + 1))
        bar = "#" * pulse + "-" * (BAR_WIDTH - pulse)
        line = f"\r{prefix} [{bar}] {format_bytes(current)} {format_speed(speed)}"

    max_len = 120
    if len(line) > max_len:
        line = line[:max_len]
    sys.stdout.write(line)
    sys.stdout.flush()


def finish_progress():
    sys.stdout.write("\n")
    sys.stdout.flush()


def get_soup(url: str):
    try:
        response = scraper.get(url, timeout=20)
        response.raise_for_status()
        response.encoding = "utf-8"
        return BeautifulSoup(response.text, "html.parser")
    except Exception as e:
        console.print(f"[bold bright_red]Error fetching page:[/bold bright_red] {e}")
        return None


def get_video_link(soup):
    if not soup:
        return None

    source_tag = soup.find("source")
    if source_tag and source_tag.get("src"):
        return source_tag.get("src")

    video_tag = soup.find("video")
    if video_tag and video_tag.get("src"):
        return video_tag.get("src")

    patterns = [
        r'const\s+videoSrc\s*=\s*"([^"]+)"',
        r"const\s+videoSrc\s*=\s*'([^']+)'",
        r'"videoSrc"\s*:\s*"([^"]+)"',
        r"'videoSrc'\s*:\s*'([^']+)'",
        r'file\s*:\s*"([^"]+\.mp4[^"]*)"',
        r"file\s*:\s*'([^']+\.mp4[^']*)'",
        r'src\s*:\s*"([^"]+\.mp4[^"]*)"',
        r"src\s*:\s*'([^']+\.mp4[^']*)'",
    ]

    for script in soup.find_all("script"):
        content = script.string or script.get_text() or ""
        for pattern in patterns:
            match = re.search(pattern, content)
            if match:
                return match.group(1)

    return None


def discover_episodes(page_url: str, soup):
    episodes = []
    selectors = [
        ("div.episode-item", "div.episode-number"),
        ("li.episode-item", ".episode-number"),
        (".episode-item", ".episode-number"),
    ]

    for item_selector, num_selector in selectors:
        items = soup.select(item_selector)
        if not items:
            continue

        for item in items:
            num_el = item.select_one(num_selector)
            link_el = item.find_parent("a") or item.find("a")
            if not num_el or not link_el or not link_el.get("href"):
                continue

            num_text = re.sub(r"\D+", "", num_el.get_text(strip=True))
            if not num_text:
                continue

            episodes.append({
                "num": int(num_text),
                "link": urljoin(page_url, link_el["href"]),
            })

        if episodes:
            break

    if not episodes:
        episodes = [{"num": 1, "link": page_url}]

    unique = {}
    for ep in episodes:
        unique[ep["num"]] = ep["link"]

    result = [{"num": k, "link": v} for k, v in unique.items()]
    result.sort(key=lambda x: x["num"])
    return result


def parse_selection(choice: str, total: int):
    choice = choice.strip().lower()
    if choice == "all":
        return list(range(1, total + 1))

    selected = set()
    for part in choice.replace(" ", "").split(","):
        if not part:
            continue
        if "-" in part:
            try:
                start, end = map(int, part.split("-", 1))
                if start > end:
                    start, end = end, start
                for i in range(start, end + 1):
                    if 1 <= i <= total:
                        selected.add(i)
            except Exception:
                continue
        else:
            try:
                num = int(part)
                if 1 <= num <= total:
                    selected.add(num)
            except Exception:
                continue

    return sorted(selected)


def looks_like_html_start(data: bytes):
    sample = data[:400].lower()
    return b"<html" in sample or b"<!doctype" in sample or b"<head" in sample


def download_video(url: str, filepath: Path, display_name: str, referer: str):
    temp_path = filepath.with_suffix(filepath.suffix + ".part")
    try:
        headers = {
            "Referer": referer,
            "Origin": "https://www.dimakids.com",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            "Accept": "*/*",
            "Accept-Language": "en-US,en;q=0.9,ar;q=0.8",
            "Connection": "keep-alive",
        }

        response = scraper.get(url, stream=True, headers=headers, timeout=90, allow_redirects=True)
        response.raise_for_status()

        content_type = (response.headers.get("content-type") or "").lower()
        total_size = int(response.headers.get("content-length", 0) or 0)

        if "text/html" in content_type:
            console.print("[bold bright_red]Download blocked:[/bold bright_red] server returned HTML instead of video.")
            return False

        filepath.parent.mkdir(parents=True, exist_ok=True)
        downloaded = 0
        first_chunk = b""
        start_time = time.time()
        last_draw = 0

        with open(temp_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                if not chunk:
                    continue

                if not first_chunk:
                    first_chunk = chunk[:400]
                    if looks_like_html_start(first_chunk):
                        f.close()
                        temp_path.unlink(missing_ok=True)
                        console.print("[bold bright_red]Blocked response:[/bold bright_red] received HTML page, not MP4.")
                        return False

                f.write(chunk)
                downloaded += len(chunk)

                now = time.time()
                if now - last_draw >= 0.15:
                    render_progress(f"Downloading {display_name}", downloaded, total_size, start_time)
                    last_draw = now

        render_progress(f"Downloading {display_name}", downloaded, total_size, start_time)
        finish_progress()

        if downloaded < 1024 * 1024:
            try:
                with open(temp_path, "rb") as small_file:
                    sample = small_file.read(400)
                if looks_like_html_start(sample):
                    temp_path.unlink(missing_ok=True)
                    console.print("[bold bright_red]Wrong file:[/bold bright_red] small HTML response saved instead of video.")
                    return False
            except Exception:
                pass

        if filepath.exists():
            filepath.unlink()
        temp_path.replace(filepath)
        return True

    except Exception as e:
        temp_path.unlink(missing_ok=True)
        console.print(f"[bold bright_red]Download error:[/bold bright_red] {e}")
        return False


def gather_all_files(folder: Path):
    files = []
    for root, _, filenames in os.walk(folder):
        root_path = Path(root)
        for filename in filenames:
            file_path = root_path / filename
            if file_path.is_file():
                files.append(file_path)
    return files


def zip_folder_with_progress(folder_path: Path, zip_path: Path):
    files = gather_all_files(folder_path)
    total_bytes = sum(file.stat().st_size for file in files)

    if zip_path.exists():
        zip_path.unlink()

    start_time = time.time()
    processed = 0
    last_draw = 0

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        if not files:
            zf.writestr(f"{folder_path.name}/", "")
        else:
            for file_path in files:
                arcname = str(file_path.relative_to(folder_path.parent))
                with open(file_path, "rb") as src, zf.open(arcname, "w") as dst:
                    while True:
                        chunk = src.read(CHUNK_SIZE)
                        if not chunk:
                            break
                        dst.write(chunk)
                        processed += len(chunk)
                        now = time.time()
                        if now - last_draw >= 0.15:
                            render_progress(f"Zipping {folder_path.name}", processed, total_bytes, start_time)
                            last_draw = now

    render_progress(f"Zipping {folder_path.name}", processed, total_bytes, start_time)
    finish_progress()
    console.print(f"[bold bright_green]✔ Success: {zip_path} created.[/bold bright_green]")
    return zip_path


def get_gofile_server():
    candidates = [
        "https://api.gofile.io/servers",
        "https://api.gofile.io/getServer",
    ]

    for endpoint in candidates:
        try:
            resp = requests.get(endpoint, timeout=15)
            resp.raise_for_status()
            data = resp.json()
            if isinstance(data, dict) and isinstance(data.get("data"), dict):
                data_block = data["data"]
                if isinstance(data_block.get("servers"), list) and data_block["servers"]:
                    first = data_block["servers"][0]
                    if isinstance(first, dict) and first.get("name"):
                        return first["name"]
                if isinstance(data_block.get("server"), str):
                    return data_block["server"]
        except Exception:
            pass

    return "store1"


def extract_gofile_link(data):
    if not isinstance(data, dict):
        return None

    if isinstance(data.get("data"), dict):
        d = data["data"]
        for key in ("downloadPage", "pageLink", "directLink"):
            value = d.get(key)
            if isinstance(value, str) and value.startswith("http"):
                return value
        if isinstance(d.get("code"), str):
            return f"https://gofile.io/d/{d['code']}"

    return None


def upload_to_gofile(file_path: Path):
    server = get_gofile_server()
    urls = [
        f"https://{server}.gofile.io/contents/uploadfile",
        f"https://{server}.gofile.io/uploadFile",
        "https://store1.gofile.io/contents/uploadfile",
        "https://store1.gofile.io/uploadFile",
    ]

    last_error = None

    for url in urls:
        try:
            with open(file_path, "rb") as f:
                encoder = MultipartEncoder(fields={"file": (file_path.name, f, "application/octet-stream")})
                start_time = time.time()
                last_draw = [0.0]

                def callback(monitor):
                    now = time.time()
                    if now - last_draw[0] >= 0.15 or monitor.bytes_read >= encoder.len:
                        render_progress(f"Uploading {file_path.name}", monitor.bytes_read, encoder.len, start_time)
                        last_draw[0] = now

                monitor = MultipartEncoderMonitor(encoder, callback)
                headers = {"Content-Type": monitor.content_type, "Accept": "application/json"}
                response = requests.post(url, data=monitor, headers=headers, timeout=3600)
                response.raise_for_status()
                data = response.json()
                render_progress(f"Uploading {file_path.name}", encoder.len, encoder.len, start_time)
                finish_progress()

                status = str(data.get("status", "")).lower()
                if status in {"ok", "success"} or "data" in data:
                    link = extract_gofile_link(data)
                    if link:
                        return link
                last_error = f"Gofile response error: {json.dumps(data, ensure_ascii=False)}"
        except Exception as e:
            finish_progress()
            last_error = str(e)

    console.print(f"[bold bright_red]Upload failed:[/bold bright_red] {last_error}")
    return None


def list_folders(base_dir: Path):
    folders = []
    for item in base_dir.iterdir():
        if not item.is_dir():
            continue
        if item.name.startswith("."):
            continue
        if item.name in HIDDEN_FOLDERS:
            continue
        folders.append(item)
    folders.sort(key=lambda x: x.name.lower())
    return folders


def choose_folder(base_dir: Path, preferred_folder: Path = None):
    folders = list_folders(base_dir)
    if not folders:
        console.print("[bold bright_red]No folders found.[/bold bright_red]")
        return None

    console.print(f"\n[bold bright_cyan]Folders in {base_dir}[/bold bright_cyan]")
    for idx, folder in enumerate(folders, 1):
        mark = "->" if preferred_folder and folder.resolve() == preferred_folder.resolve() else "  "
        console.print(f"[bright_white]{mark} {idx} : {folder.name}[/bright_white]")

    console.print("\n[bright_yellow]Choose folder number, full path, or press Enter to use the arrow folder.[/bright_yellow]")
    choice = Prompt.ask("[bold bright_cyan]Folder[/bold bright_cyan]", default="").strip()

    if choice == "" and preferred_folder and preferred_folder.exists():
        return preferred_folder

    if choice.isdigit():
        index = int(choice) - 1
        if 0 <= index < len(folders):
            return folders[index]

    selected = Path(choice)
    if selected.exists() and selected.is_dir():
        return selected

    console.print("[bold bright_red]Invalid folder selection.[/bold bright_red]")
    return None


def process_page(url: str):
    soup = get_soup(url)
    if not soup:
        return

    title_tag = soup.find("h1", class_="series-title") or soup.find("h1")
    title = clean_name(title_tag.get_text(strip=True) if title_tag else "Video")
    episodes = discover_episodes(url, soup)

    console.print(f"\n[bold bright_green]Title:[/bold bright_green] {title}")
    console.print(f"[bold bright_cyan]Total Episodes Found:[/bold bright_cyan] {len(episodes)}")

    choice = Prompt.ask("[bold bright_cyan]Enter episodes to download (e.g. 1,3-5,all)[/bold bright_cyan]")
    selected_nums = parse_selection(choice, len(episodes))

    if not selected_nums:
        console.print("[bold bright_red]No valid episodes selected.[/bold bright_red]")
        return

    output_folder = BASE_DIR / title
    output_folder.mkdir(parents=True, exist_ok=True)

    for num in selected_nums:
        ep = next((e for e in episodes if e["num"] == num), None)
        if not ep:
            continue

        console.print(f"\n[bold yellow]Processing Episode {num}...[/bold yellow]")
        ep_soup = get_soup(ep["link"])
        video_link = get_video_link(ep_soup)

        if not video_link:
            console.print(f"[bold bright_red]Link not found for Episode {num}[/bold bright_red]")
            continue

        file_path = output_folder / f"{num}.mp4"
        ok = download_video(video_link, file_path, f"{num}.mp4", ep["link"])
        if ok:
            console.print(f"[bold bright_green]Saved:[/bold bright_green] {file_path.name}")

    console.print(f"\n[bold bright_green]Download folder:[/bold bright_green] {output_folder.name}")

    selected_folder = choose_folder(BASE_DIR, preferred_folder=output_folder)
    if not selected_folder:
        return

    zip_path = BASE_DIR / f"{selected_folder.name}.zip"
    zip_folder_with_progress(selected_folder, zip_path)

    link = upload_to_gofile(zip_path)
    if link:
        console.print("\n[bold bright_green]Link Generated Successfully![/bold bright_green]")
        console.print(f"[bold bright_white on green] Download Link: {link} [/bold bright_white on green]")
    else:
        console.print("[bold bright_red]Could not get link. Please try again later.[/bold bright_red]")


def main():
    console.print(f"[bold bright_cyan]Base folder:[/bold bright_cyan] {BASE_DIR}")
    while True:
        url = Prompt.ask("\n[bold bright_cyan]DimaKids URL (q to quit)[/bold bright_cyan]").strip()
        if url.lower() == "q":
            break
        if not url.startswith("http"):
            console.print("[bold bright_red]Please enter a valid URL.[/bold bright_red]")
            continue
        process_page(url)


if __name__ == "__main__":
    main()



Base folder: /content

DimaKids URL (q to quit):

https://www.dimakids.com/made_in_abyss-1740913479-anime-streaming.html


Title: صنع في الهاوية

Total Episodes Found: 13

...

Uploading صنع في الهاوية.zip [##################] 100.0% 145.4MB/145.4MB 16.0MB/s ETA 00:00


Link Generated Successfully!

 Download Link: https://gofile.io/d/8eQEQj 

DimaKids URL (q to quit):

q
